<a href="https://colab.research.google.com/github/Arif0000/GFG-21-Days-Live-class-task/blob/main/Task_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install kagglehub pandas google-generativeai


import kagglehub
import os

path = kagglehub.dataset_download("snehaanbhawal/resume-dataset")

print("Dataset path:", path)
print("Folders:", os.listdir(path))



csv_file = None

for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith(".csv"):
            csv_file = os.path.join(root, file)

print("Found CSV file at:", csv_file)



import pandas as pd

df = pd.read_csv(csv_file)


df.columns = df.columns.str.strip().str.lower()

print("\nColumns:", df.columns)
print(df.head())



resume_col = None

for col in df.columns:
    if "resume" in col:
        resume_col = col
        break

if resume_col is None:
    raise Exception("Resume column not found!")

print("Using resume column:", resume_col)



import google.generativeai as genai

genai.configure(api_key="Enter yous gemini API key here")

model = genai.GenerativeModel("gemini-2.5-flash")



base_prompt = """
Extract the following information from the resume:

1. Name
2. Skills (as a list)
3. Education
4. Experience

Return ONLY JSON:

{
    "name": "",
    "skills": [],
    "education": "",
    "experience": ""
}

Resume text:
"""



import json
import time

results = []

num_samples = 10  # change if needed

for i in range(num_samples):

    print(f"\nProcessing Resume {i+1}/{num_samples}")

    resume_text = str(df.loc[i, resume_col])[:3000]

    prompt = base_prompt + resume_text

    try:
        response = model.generate_content(prompt)

        clean_text = response.text.replace("```json", "").replace("```", "")
        data = json.loads(clean_text)

    except Exception as e:
        print("Error:", e)
        data = {"error": "Parsing failed"}

    results.append(data)

    time.sleep(5)  # avoid rate limits



output_path = os.path.join(path, "resume_output.json")

with open(output_path, "w") as f:
    json.dump(results, f, indent=4)

print("\nExtraction completed!")
print("Saved at:", output_path)



job_skills = ["Python", "Machine Learning", "SQL"]

def match_skills(candidate_skills):
    if isinstance(candidate_skills, list):
        return list(set(candidate_skills) & set(job_skills))
    return []

print("\nSkill Match Example:")
print(match_skills(results[0].get("skills", [])))